Tested on windows based on this wonderful [video](https://www.youtube.com/watch?v=6oGbsAg8x5E) need to install gemma 2b on ollama like in the video.

Also, download the Trial by Sorcery file from this [site](https://manybooks.net/titles/trial-by-sorcer)

You need faiss-gpu and a T4 in Collab for an acceptable performance.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "10" #change accordingly

In [2]:
# Import necessary libraries                                                                                                                                                                                                                    
from langchain_huggingface.embeddings import HuggingFaceEmbeddings                                                                                                                                                                   
from langchain_community.vectorstores import FAISS                                                                                                                                                                                                                                                                                                                                                                                                
from langchain_community.document_loaders import PyPDFLoader                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

loader = PyPDFLoader("./Trial-by-Sorcery.pdf")
pages = loader.load_and_split()
                                                                                                                                                                                                                                                                                                                                       
model_id = "nomic-ai/nomic-embed-text-v1"
model_kwargs = {'device': 'cpu'}
embeddings = HuggingFaceEmbeddings(
    model_name=model_id,
    model_kwargs=model_kwargs
)


faiss_index = FAISS.from_documents(pages, embeddings)


c:\work\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 112/112 [00:00<00:00, 8349.25it/s]


In [3]:
inquiry = "Who is Eldwin's father?"  
docs = faiss_index.similarity_search(inquiry, k=5)
for doc in docs:
    print(str(doc.metadata["page"]) + ":", doc.page_content[:300]) 

66: “I don’t think so?”
“How do you not know, Eldwin? The battle your father died in was
against the False King.”
I’d honestly never heard the name before, but suddenly knowing that
he was the cause of my father’s death made me hate him, and I didn’t
even know who he was.
“Who’s the False King?” I asked
38: Simon turned to the left, away from the market and down an alley. I
wasn’t sure where he was leading me, but I continued to follow him.
Another left took us behind a building. There was a group of guards
leaning against the building’s wall. A feeling of uneasiness swept over
me as I saw the men were
132: humans began to domesticate dragons, they stopped laying their eggs
here.”
“I’ve never heard that before,” I said.
“This place holds a lot more secrets, though many of them have been
lost to the centuries. Tell me, Eldwin. Did your father tell you the color
of his dragon?”
“He rode a blue,” I said.

12: “I know your surname, but what’s your given name?”
“Eldwin,” I said.
The guard n

In [4]:
def retrieve_context_plain(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = faiss_index.similarity_search(query, k=2)
    return "\n".join(doc.page_content for doc in retrieved_docs)

retrieve_context_plain(inquiry)

'“I don’t think so?”\n“How do you not know, Eldwin? The battle your father died in was\nagainst the False King.”\nI’d honestly never heard the name before, but suddenly knowing that\nhe was the cause of my father’s death made me hate him, and I didn’t\neven know who he was.\n“Who’s the False King?” I asked.\nSimon turned to the left, away from the market and down an alley. I\nwasn’t sure where he was leading me, but I continued to follow him.\nAnother left took us behind a building. There was a group of guards\nleaning against the building’s wall. A feeling of uneasiness swept over\nme as I saw the men were city guards.\nI started to backstep, but the clink of chainmail alerted me to more\nguards closing in from behind. I instinctively reached for my sword but\nremembered it was in the Citadel’s armory. My heart started hammering\nin my chest.\n“Well done, Simon,” one of the guards said. “I was starting to think\nhe had left the city before I could give him a proper introduction to\nAu

In [5]:
from langchain_ollama import ChatOllama

# Suitable for tool calling https://ollama.com/search?&c=tools
model = ChatOllama(model="qwen3.5:2b")                                                                                                                                                                                                                                                                                                                                                                      

In [6]:
some_prompt = f"{retrieve_context_plain(inquiry)} \n {inquiry}"

some_answer = model.invoke(some_prompt)

print(some_answer)

content='According to the text provided, Eldwin\'s father is **Matthias Baines**.\n\nThis is stated during the dialogue between the guard and Eldwin:\n\n> "Eldwin\'s father was Matthias Baines."' additional_kwargs={} response_metadata={'model': 'qwen3.5:2b', 'created_at': '2026-04-06T14:20:01.19543Z', 'done': True, 'done_reason': 'stop', 'total_duration': 68891046200, 'load_duration': 250017900, 'prompt_eval_count': 493, 'prompt_eval_duration': 10274845900, 'eval_count': 436, 'eval_duration': 57922046900, 'logprobs': None, 'model_name': 'qwen3.5:2b', 'model_provider': 'ollama'} id='lc_run--019d6329-1e0d-7a90-bd1a-2c2bc8c9e590-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 493, 'output_tokens': 436, 'total_tokens': 929}
